# TF-IDF とロジスティック回帰で日本語のニュース記事を分類する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_classification/tfidf_logreg_ja.ipynb)

NLP クックブック 14-01 ／ テキスト分類

日本語の文書分類を、まずは軽くて強い古典的な手法でやってみたい。livedoor ニュースコーパス (9 カテゴリ) を使う。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, tarfile, urllib.request
import pandas as pd

def load_livedoor():
    """livedoor ニュースコーパス (CC BY-ND) を DataFrame にする"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")  # 1 行目 URL, 2 行目 日付, 3 行目 タイトル
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
print(len(df))
df["label"].value_counts()

7367


label
sports-watch      900
smax              870
movie-enter       870
it-life-hack      870
dokujo-tsushin    870
kaden-channel     864
peachy            842
topic-news        770
livedoor-homme    511
Name: count, dtype: int64

In [3]:
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

tagger = Tagger("-Owakati")
df["wakati"] = [tagger.parse(t[:2000]).strip() for t in df["text"]]   # 先頭 2000 文字を分かち書き

train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
clf = make_pipeline(
    TfidfVectorizer(token_pattern=r"\S+", min_df=3, max_df=0.5, sublinear_tf=True),   # 分かち書き済みなので空白で区切る
    LogisticRegression(C=10, max_iter=1000),
)
clf.fit(train["wakati"], train["label"])
print("正解率:", accuracy_score(test["label"], clf.predict(test["wakati"])))

正解率: 0.9491180461329715


In [4]:
new = "日本代表は後半に 2 点を奪って逆転し、ワールドカップ予選で勝利を収めた。"
print(clf.predict([tagger.parse(new)]), clf.predict_proba([tagger.parse(new)]).max().round(2))

['sports-watch'] 0.92


- 先に分かち書きして空白区切りの文字列にしておくと、scikit-learn の標準的な部品がそのまま使える。
- このベースラインで 9 割を超える。BERT を持ち出す前に、まず TF-IDF + 線形モデルを試すのが定石。